# 🌿 Domain-Generalized Plant Disease Recognition: Full GPU Training
### State-of-the-Art Research Capstone: ResNet50 + MixStyle (ICLR 2021) on Google Colab (T4 GPU)

---
### 📌 Overview & Target Specifications
This notebook trains the **ResNet50 + MixStyle Domain Generalization (DG)** model across all **30,824 images (28 classes)** for **15 full epochs** with:
- ⚡ **Hardware Acceleration:** NVIDIA T4 GPU (15GB VRAM) with `mixed_float16` precision policy (utilizes Tensor Cores, doubles throughput and reduces VRAM usage).
- 🔬 **Optimization Suite:**
  - **Optimizer:** `AdamW` (learning_rate = 1e-4, weight_decay = 1e-4)
  - **Loss Function:** `CategoricalCrossentropy` with **Label Smoothing (alpha = 0.1)** to prevent overconfident calibration.
  - **Weighted Loss:** Balanced inverse-frequency class weights ($w_c$) to combat dataset skew.
- 🛡️ **Regularization & Generalization:**
  - **MixStyle Layer:** In-network instance feature statistics mixing during forward pass.
  - **MixUp Data Augmentation:** Batch-level linear convex combinations (alpha = 0.2).
- ⏱️ **Callbacks:**
  - `ModelCheckpoint`: Saves best weights to `/content/best_model.keras` based on minimum validation loss (`val_loss`).
  - `EarlyStopping`: Halts training if validation loss does not improve for 5 epochs (patience = 5).
  - `ReduceLROnPlateau`: Halves learning rate on validation loss stagnation (patience = 3, factor = 0.5, min_lr = 1e-6).
  - `TensorBoard`: Logs scalar metrics and learning rate schedules to `/content/logs/tensorboard/`.
  - `CSVLogger`: Logs epoch metrics to `/content/training_history.csv`.
- 💾 **One-Click Export:** Auto-downloads `best_model.keras` directly to your local computer.

**⏱️ Expected Training Duration:** ~18 to 22 minutes on Colab Free T4 GPU.

> **Step 1:** In Google Colab top menu, go to **Runtime ➔ Change runtime type ➔ Select 'T4 GPU' ➔ Save**.
> **Step 2:** Click **Runtime ➔ Run all** (`Ctrl + F9`).

## 1. Hardware Verification & Mixed Precision Policy
Verifies NVIDIA GPU and enables `mixed_float16` to leverage Tensor Cores.

In [ ]:
!nvidia-smi

import os
import sys
import json
from pathlib import Path
import numpy as np
import pandas as pd
import tensorflow as tf
import keras

print(f"TensorFlow Version: {tf.__version__}")
print(f"Keras Version:      {keras.__version__}")

# Enable mixed_float16 precision policy for NVIDIA Tensor Cores
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        try:
            tf.config.experimental.set_memory_growth(gpu, True)
        except Exception:
            pass
    keras.mixed_precision.set_global_policy('mixed_float16')
    print(f"[SUCCESS] {len(gpus)} GPU Detected -> 'mixed_float16' policy active on Tensor Cores!")
else:
    print("[WARN] No GPU detected. Please go to Runtime -> Change runtime type -> T4 GPU.")

## 2. Dataset Acquisition & Setup in Colab
This cell automatically acquires the **PlantVillage** dataset in Colab.
- If you mounted Google Drive, it checks `/content/drive/MyDrive`.
- Otherwise, it automatically clones the official PlantVillage repository (~40-60 seconds on Colab's high-speed connection).

In [ ]:
# Create directories for output
!mkdir -p /content/models/saved_models
!mkdir -p /content/results/figures
!mkdir -p /content/results/metrics
!mkdir -p /content/logs/tensorboard

# Check if Google Drive is mounted and dataset exists
dataset_dir = None
drive_candidate = Path("/content/drive/MyDrive/plantvillage")
local_candidate = Path("/content/plantvillage")

if drive_candidate.exists():
    dataset_dir = drive_candidate
    print(f"[FOUND] Dataset found in Google Drive: {dataset_dir}")
elif local_candidate.exists():
    dataset_dir = local_candidate
    print(f"[FOUND] Dataset found locally: {dataset_dir}")
else:
    print("[DOWNLOAD] Downloading PlantVillage dataset from official repository...")
    # Option A: Fast clone of official PlantVillage dataset
    !git clone --depth 1 https://github.com/spMohanty/PlantVillage-Dataset.git /content/plantvillage_repo
    dataset_dir = Path("/content/plantvillage_repo/raw/color")

print(f"Dataset Directory: {dataset_dir}")
assert dataset_dir.exists(), f"Dataset directory {dataset_dir} does not exist!"

## 3. Canonical 28-Class Taxonomy & Balanced Class Weights

In [ ]:
CANONICAL_CLASSES = [
    "apple_apple_scab",
    "apple_cedar_apple_rust",
    "apple_healthy",
    "bell_pepper_bacterial_spot",
    "bell_pepper_healthy",
    "blueberry_healthy",
    "cherry_healthy",
    "corn_gray_leaf_spot",
    "corn_leaf_blight",
    "corn_rust",
    "grape_black_rot",
    "grape_healthy",
    "peach_healthy",
    "potato_early_blight",
    "potato_late_blight",
    "raspberry_healthy",
    "soybean_healthy",
    "squash_powdery_mildew",
    "strawberry_healthy",
    "tomato_bacterial_spot",
    "tomato_early_blight",
    "tomato_healthy",
    "tomato_late_blight",
    "tomato_leaf_mold",
    "tomato_mosaic_virus",
    "tomato_septoria_leaf_spot",
    "tomato_spider_mites",
    "tomato_yellow_leaf_curl_virus",
]
NUM_CLASSES = len(CANONICAL_CLASSES)  # 28
CANONICAL_TO_ID = {name: idx for idx, name in enumerate(CANONICAL_CLASSES)}

# Map PlantVillage directory names to canonical classes
PLANTVILLAGE_TO_CANONICAL = {
    "Apple___Apple_scab": "apple_apple_scab",
    "Apple___Cedar_apple_rust": "apple_cedar_apple_rust",
    "Apple___healthy": "apple_healthy",
    "Pepper,_bell___Bacterial_spot": "bell_pepper_bacterial_spot",
    "Pepper,_bell___healthy": "bell_pepper_healthy",
    "Blueberry___healthy": "blueberry_healthy",
    "Cherry_(including_sour)___healthy": "cherry_healthy",
    "Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot": "corn_gray_leaf_spot",
    "Corn_(maize)___Northern_Leaf_Blight": "corn_leaf_blight",
    "Corn_(maize)___Common_rust_": "corn_rust",
    "Grape___Black_rot": "grape_black_rot",
    "Grape___healthy": "grape_healthy",
    "Peach___healthy": "peach_healthy",
    "Potato___Early_blight": "potato_early_blight",
    "Potato___Late_blight": "potato_late_blight",
    "Raspberry___healthy": "raspberry_healthy",
    "Soybean___healthy": "soybean_healthy",
    "Squash___Powdery_mildew": "squash_powdery_mildew",
    "Strawberry___healthy": "strawberry_healthy",
    "Tomato___Bacterial_spot": "tomato_bacterial_spot",
    "Tomato___Early_blight": "tomato_early_blight",
    "Tomato___healthy": "tomato_healthy",
    "Tomato___Late_blight": "tomato_late_blight",
    "Tomato___Leaf_Mold": "tomato_leaf_mold",
    "Tomato___Tomato_mosaic_virus": "tomato_mosaic_virus",
    "Tomato___Septoria_leaf_spot": "tomato_septoria_leaf_spot",
    "Tomato___Spider_mites Two-spotted_spider_mite": "tomato_spider_mites",
    "Tomato___Tomato_Yellow_Leaf_Curl_Virus": "tomato_yellow_leaf_curl_virus",
}

# Precomputed balanced class weights w_c = N_total / (C * N_c) from Phase 2
CLASS_WEIGHTS = {
    0: 2.1849, 1: 5.0055, 2: 0.8368, 3: 1.3806, 4: 0.9313, 5: 0.9164,
    6: 1.6118, 7: 2.6832, 8: 1.3975, 9: 1.1548, 10: 1.1665, 11: 3.2541,
    12: 3.8236, 13: 1.3765, 14: 1.3765, 15: 3.7102, 16: 0.2704, 17: 0.7501,
    18: 3.0186, 19: 0.6472, 20: 1.3765, 21: 0.8652, 22: 0.7211, 23: 1.4459,
    24: 3.6903, 25: 0.7772, 26: 0.8213, 27: 0.2570
}
print(f"Taxonomy Initialized: {NUM_CLASSES} Canonical Classes | {len(CLASS_WEIGHTS)} Balanced Weights Ready.")

## 4. Dataset Indexing & High-Performance `tf.data` Pipeline
Performs stratified 80/20 train/validation split, ImageNet normalization, and MixUp augmentation.

In [ ]:
from sklearn.model_selection import train_test_split

# 1. Collect all valid image paths belonging to canonical classes
image_paths = []
image_labels = []

for folder in dataset_dir.iterdir():
    if folder.is_dir() and folder.name in PLANTVILLAGE_TO_CANONICAL:
        canonical_name = PLANTVILLAGE_TO_CANONICAL[folder.name]
        class_id = CANONICAL_TO_ID[canonical_name]
        for ext in ["*.jpg", "*.JPG", "*.jpeg", "*.png", "*.PNG"]:
            for img_file in folder.glob(ext):
                image_paths.append(str(img_file))
                image_labels.append(class_id)

print(f"Total Canonical Images Found: {len(image_paths):,}")

# 2. Stratified 80% Train, 20% Val split
train_paths, val_paths, train_lbls, val_lbls = train_test_split(
    image_paths, image_labels,
    test_size=0.20,
    stratify=image_labels,
    random_state=42,
)

print(f"Training Split:   {len(train_paths):,} images")
print(f"Validation Split: {len(val_paths):,} images")

# 3. tf.data Dataset Pipeline
IMAGENET_MEAN = tf.constant([0.485, 0.456, 0.406], dtype=tf.float32)
IMAGENET_STD = tf.constant([0.229, 0.224, 0.225], dtype=tf.float32)

def parse_image_and_label(path, label, is_training=False):
    img_bytes = tf.io.read_file(path)
    img = tf.io.decode_image(img_bytes, channels=3, expand_animations=False)
    img.set_shape([None, None, 3])
    img = tf.image.resize(img, (224, 224), method="bilinear")
    
    if is_training:
        img = tf.image.random_flip_left_right(img)
        img = tf.image.random_flip_up_down(img)
        img = tf.image.random_brightness(img, max_delta=25.0)
        img = tf.image.random_contrast(img, lower=0.8, upper=1.2)
        img = tf.image.random_saturation(img, lower=0.8, upper=1.2)
        img = tf.image.random_hue(img, max_delta=0.05)
        
    img = tf.cast(img, tf.float32) / 255.0
    img = (img - IMAGENET_MEAN) / IMAGENET_STD
    one_hot = tf.one_hot(label, depth=NUM_CLASSES)
    return img, one_hot

def mixup(images, labels, alpha=0.2):
    batch_size = tf.shape(images)[0]
    g1 = tf.random.gamma(shape=[batch_size, 1, 1, 1], alpha=alpha)
    g2 = tf.random.gamma(shape=[batch_size, 1, 1, 1], alpha=alpha)
    lam_img = g1 / (g1 + g2)
    lam_lbl = tf.reshape(lam_img, [batch_size, 1])
    
    indices = tf.random.shuffle(tf.range(batch_size))
    mixed_images = (lam_img * images) + ((1.0 - lam_img) * tf.gather(images, indices))
    mixed_labels = (lam_lbl * labels) + ((1.0 - lam_lbl) * tf.gather(labels, indices))
    return mixed_images, mixed_labels

BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

train_ds = tf.data.Dataset.from_tensor_slices((train_paths, train_lbls))
train_ds = train_ds.shuffle(buffer_size=2048, seed=42, reshuffle_each_iteration=True)
train_ds = train_ds.map(lambda p, l: parse_image_and_label(p, l, is_training=True), num_parallel_calls=AUTOTUNE)
train_ds = train_ds.batch(BATCH_SIZE, drop_remainder=True)
train_ds = train_ds.map(mixup, num_parallel_calls=AUTOTUNE)
train_ds = train_ds.prefetch(AUTOTUNE)

val_ds = tf.data.Dataset.from_tensor_slices((val_paths, val_lbls))
val_ds = val_ds.map(lambda p, l: parse_image_and_label(p, l, is_training=False), num_parallel_calls=AUTOTUNE)
val_ds = val_ds.batch(BATCH_SIZE)
val_ds = val_ds.prefetch(AUTOTUNE)

print("High-Performance tf.data Input Pipeline constructed successfully!")

## 5. Custom MixStyle Domain Generalization Layer (ICLR 2021)
Mixes feature statistics (mean and variance) across batch samples to simulate unseen target domains.

In [ ]:
@keras.saving.register_keras_serializable(package="PlantDG", name="MixStyle")
class MixStyle(keras.layers.Layer):
    def __init__(self, p: float = 0.5, alpha: float = 0.1, eps: float = 1e-6, **kwargs):
        super().__init__(**kwargs)
        self.p = float(p)
        self.alpha = float(alpha)
        self.eps = float(eps)

    def call(self, x: tf.Tensor, training: bool = None) -> tf.Tensor:
        if not training:
            return x

        should_apply = tf.random.uniform([]) < self.p

        def mix_features():
            batch_size = tf.shape(x)[0]
            mu = tf.reduce_mean(x, axis=[1, 2], keepdims=True)
            var = tf.math.reduce_variance(x, axis=[1, 2], keepdims=True)
            sig = tf.sqrt(var + self.eps)
            norm_x = (x - mu) / sig

            # XLA and Tensor-Core compatible Beta(alpha, alpha) sampling
            u = tf.random.uniform([batch_size, 1, 1, 1], minval=1e-6, maxval=1.0 - 1e-6, dtype=tf.float32)
            inv_alpha = 1.0 / float(self.alpha)
            half = tf.constant(0.5, dtype=tf.float32)
            lam_f32 = tf.where(
                u < half,
                half * tf.pow(2.0 * u, inv_alpha),
                1.0 - half * tf.pow(2.0 * (1.0 - u), inv_alpha)
            )
            lam = tf.cast(lam_f32, dtype=x.dtype)
            one = tf.cast(1.0, dtype=x.dtype)

            perm = tf.random.shuffle(tf.range(batch_size))
            mu_shuffled = tf.gather(mu, perm)
            sig_shuffled = tf.gather(sig, perm)

            mu_mix = (lam * mu) + ((one - lam) * mu_shuffled)
            sig_mix = (lam * sig) + ((one - lam) * sig_shuffled)
            return (sig_mix * norm_x) + mu_mix

        return tf.cond(should_apply, mix_features, lambda: x)

    def get_config(self):
        config = super().get_config()
        config.update({"p": self.p, "alpha": self.alpha, "eps": self.eps})
        return config

    @classmethod
    def from_config(cls, config):
        return cls(**config)

print("MixStyle Layer defined & registered successfully (100% XLA & GPU Compatible)!")

## 6. Build ResNet50 + MixStyle Architecture

In [ ]:
from keras.src.applications import resnet as resnet_lib

def build_resnet50_mixstyle(num_classes=28, input_shape=(224, 224, 3), p=0.5, alpha=0.1):
    inputs = keras.Input(shape=input_shape, name="input_image")

    # Stem
    x = keras.layers.ZeroPadding2D(padding=((3, 3), (3, 3)), name="conv1_pad")(inputs)
    x = keras.layers.Conv2D(64, 7, strides=2, use_bias=True, name="conv1_conv")(x)
    x = keras.layers.BatchNormalization(axis=3, epsilon=1.001e-5, name="conv1_bn")(x)
    x = keras.layers.Activation("relu", name="conv1_relu")(x)
    x = keras.layers.ZeroPadding2D(padding=((1, 1), (1, 1)), name="pool1_pad")(x)
    x = keras.layers.MaxPooling2D(3, strides=2, name="pool1_pool")(x)

    # Stage 1: conv2 + MixStyle 1
    x = resnet_lib.stack_residual_blocks_v1(x, 64, 3, stride1=1, name="conv2")
    x = MixStyle(p=p, alpha=alpha, name="mixstyle_stage1")(x)

    # Stage 2: conv3 + MixStyle 2
    x = resnet_lib.stack_residual_blocks_v1(x, 128, 4, name="conv3")
    x = MixStyle(p=p, alpha=alpha, name="mixstyle_stage2")(x)

    # Stage 3: conv4
    x = resnet_lib.stack_residual_blocks_v1(x, 256, 6, name="conv4")

    # Stage 4: conv5 (Target feature map for Grad-CAM)
    x = resnet_lib.stack_residual_blocks_v1(x, 512, 3, name="conv5")

    # Classification Head
    x = keras.layers.GlobalAveragePooling2D(name="avg_pool")(x)
    x = keras.layers.BatchNormalization(name="head_bn")(x)
    x = keras.layers.Dense(512, activation="relu", name="head_dense")(x)
    x = keras.layers.Dropout(0.4, name="head_dropout")(x)
    outputs = keras.layers.Dense(num_classes, activation="softmax", dtype="float32", name="classification_head")(x)

    model = keras.Model(inputs=inputs, outputs=outputs, name="resnet50_mixstyle_dg")

    # Load pretrained ImageNet weights into backbone layers
    weights_path = keras.utils.get_file(
        "resnet50_weights_tf_dim_ordering_tf_kernels_notop.h5",
        "https://storage.googleapis.com/tensorflow/keras-applications/resnet/resnet50_weights_tf_dim_ordering_tf_kernels_notop.h5",
        cache_subdir="models",
    )
    model.load_weights(weights_path, by_name=True)
    print("Pretrained ImageNet weights loaded into ResNet50 + MixStyle!")
    return model

model = build_resnet50_mixstyle(num_classes=NUM_CLASSES)
print(f"Total Model Parameters: {model.count_params():,}")

## 7. Optimization Suite & Production Callbacks
Compiles with:
- **AdamW** (learning_rate = 1e-4, weight_decay = 1e-4)
- **CategoricalCrossentropy** with **Label Smoothing (alpha = 0.1)**
- **ModelCheckpoint** monitoring `val_loss` (saving to `/content/best_model.keras`)
- **EarlyStopping** (patience = 5)
- **ReduceLROnPlateau** (patience = 3, factor = 0.5, min_lr = 1e-6)
- **TensorBoard** & **CSVLogger**

In [ ]:
optimizer = keras.optimizers.AdamW(learning_rate=1e-4, weight_decay=1e-4)
loss_fn = keras.losses.CategoricalCrossentropy(label_smoothing=0.1)
top3_acc = tf.keras.metrics.TopKCategoricalAccuracy(k=3, name="top_3_accuracy")

model.compile(
    optimizer=optimizer,
    loss=loss_fn,
    metrics=["accuracy", top3_acc],
    jit_compile=False,
)

best_model_path = "/content/best_model.keras"
history_csv_path = "/content/training_history.csv"
tb_log_dir = "/content/logs/tensorboard"

callbacks = [
    keras.callbacks.ModelCheckpoint(
        filepath=best_model_path,
        monitor="val_loss",
        save_best_only=True,
        mode="min",
        verbose=1,
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
        mode="min",
        verbose=1,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        mode="min",
        verbose=1,
    ),
    keras.callbacks.CSVLogger(history_csv_path),
    keras.callbacks.TensorBoard(log_dir=tb_log_dir),
]

print("Model compiled with AdamW + Label Smoothing (0.1) & Callbacks configured!")

## 8. Execute Full 15-Epoch Training Loop on GPU 🚀
Trains across all 30,824 images for 15 epochs with balanced class weights and mixed precision.

In [ ]:
EPOCHS = 15
print(f"Starting Full Training Run for {EPOCHS} Epochs on GPU...")

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    class_weight=CLASS_WEIGHTS,
    callbacks=callbacks,
    verbose=1,
)

print("\n[DONE] Full 15-Epoch Training Complete!")
print(f"Best model saved to: {best_model_path}")

## 9. Plot Convergence Dynamics & Training Curves

In [ ]:
import matplotlib.pyplot as plt

history_dict = history.history
epochs_range = range(1, len(history_dict["loss"]) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss
ax1.plot(epochs_range, history_dict["loss"], "b-o", label="Train Loss", linewidth=2)
ax1.plot(epochs_range, history_dict["val_loss"], "r--s", label="Val Loss", linewidth=2)
ax1.set_title("Convergence Dynamics: Loss (Label Smoothed)", fontsize=12, fontweight="bold")
ax1.set_xlabel("Epoch", fontsize=10)
ax1.set_ylabel("Categorical Crossentropy", fontsize=10)
ax1.grid(True, linestyle="--", alpha=0.5)
ax1.legend(fontsize=10)

# Accuracy
ax2.plot(epochs_range, history_dict["accuracy"], "b-o", label="Train Accuracy", linewidth=2)
ax2.plot(epochs_range, history_dict["val_accuracy"], "r--s", label="Val Accuracy", linewidth=2)
if "val_top_3_accuracy" in history_dict:
    ax2.plot(epochs_range, history_dict["val_top_3_accuracy"], "g-.^", label="Val Top-3 Accuracy", linewidth=1.5)
ax2.set_title("Diagnostic Accuracy Dynamics", fontsize=12, fontweight="bold")
ax2.set_xlabel("Epoch", fontsize=10)
ax2.set_ylabel("Accuracy", fontsize=10)
ax2.set_ylim(0, 1.02)
ax2.grid(True, linestyle="--", alpha=0.5)
ax2.legend(fontsize=10)

plt.suptitle("ResNet50 + MixStyle: Full 15-Epoch Training Curves", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
curves_plot_path = "/content/resnet50_mixstyle_training_curves.png"
fig.savefig(curves_plot_path, dpi=200, bbox_inches="tight")
plt.show()
print(f"Training curves saved to: {curves_plot_path}")

## 10. Download Trained Model Weights (`best_model.keras`) 📥
Automatically downloads the 296 MB high-precision trained model directly to your browser!

In [ ]:
from google.colab import files

print("Triggering browser download for best_model.keras...")
files.download("/content/best_model.keras")

if os.path.exists("/content/training_history.csv"):
    files.download("/content/training_history.csv")

if os.path.exists("/content/resnet50_mixstyle_training_curves.png"):
    files.download("/content/resnet50_mixstyle_training_curves.png")

print("\n========================================================================")
print("[SUCCESS] Download completed! Move 'best_model.keras' into:")
print("          Deep_Learning_Project/models/best_model.keras")
print("========================================================================")